# Initialization

In [2]:
import json
import uuid
import os
import json
from dotenv import load_dotenv
from pathlib import Path
from kafka import KafkaProducer
from faker import Faker
from time import sleep

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Dibimbing Spark-Kafka") 
    .config("spark.streaming.stopGracefullyOnShutdown", True) 
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.2')
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]") 
    .getOrCreate()
)

spark

# Spark - Kafka Streaming

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window, sum as _sum, date_trunc
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType, TimestampType

# Start Spark Session
spark = SparkSession.builder \
    .appName("SimpleKafkaStream") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Define schema
schema = StructType() \
    .add("event_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("product", StringType()) \
    .add("category", StringType()) \
    .add("price", DoubleType()) \
    .add("quantity", IntegerType()) \
    .add("location", StringType()) \
    .add("ts", StringType())  # Assume Unix timestamp (string)

# Read Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "purchasing_events") \
    .option("startingOffsets", "latest") \
    .load()

# Parse and cast event time
events = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("ts").cast("timestamp"))

# Add truncated hour to use as a grouping key
events = events.withWatermark("event_time", "1 hour") \
    .withColumn("hour_bucket", date_trunc("hour", col("event_time")))

# 5-minute window aggregation (with hourly running total grouping)
agg = events.groupBy(
    window(col("event_time"), "5 minutes"),
    col("hour_bucket")
).agg(
    _sum(col("price") * col("quantity")).alias("hourly_total")
).select(
    col("window.end").alias("timestamp"),
    col("hourly_total")
)

# Output
query = agg.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

query.awaitTermination()


AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        

In [3]:
dotenv_path = Path('/resources/.env')
load_dotenv(dotenv_path=dotenv_path)

True

In [4]:
kafka_host = os.getenv('KAFKA_HOST')
kafka_topic = os.getenv('KAFKA_TOPIC_NAME')
kafka_topic_partition = os.getenv('KAFKA_TOPIC_NAME')+"-1"

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window, sum as _sum
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType, TimestampType

# Start Spark Session
spark = SparkSession.builder \
    .appName("PurchasingEventConsumer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Kafka topic config
kafka_topic = "purchasing_events"
kafka_bootstrap_servers = "localhost:9092"

# Define schema of incoming JSON
purchase_schema = StructType() \
    .add("event_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("product", StringType()) \
    .add("category", StringType()) \
    .add("price", DoubleType()) \
    .add("quantity", IntegerType()) \
    .add("location", StringType()) \
    .add("ts", StringType())  # We'll cast to timestamp

# Read stream from Kafka
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .load()

# Parse JSON and cast event time
parsed_stream = raw_stream.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), purchase_schema).alias("data")) \
    .selectExpr("data.*") \
    .withColumn("event_time", col("ts").cast(TimestampType()))

# Watermark + 5-minute aggregation
five_minute_agg = parsed_stream \
    .withWatermark("event_time", "1 hour") \
    .groupBy(window(col("event_time"), "5 minutes")) \
    .agg(_sum(col("price") * col("quantity")).alias("total_purchase")) \
    .selectExpr("window.start as timestamp", "total_purchase")

# Watermark + hourly aggregation
hourly_total = parsed_stream \
    .withWatermark("event_time", "1 hour") \
    .groupBy(window(col("event_time"), "1 hour")) \
    .agg(_sum(col("price") * col("quantity")).alias("hourly_total")) \
    .selectExpr("window.end as timestamp", "hourly_total")

# Write 5-min aggregation to console
five_min_query = five_minute_agg.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .queryName("five_min_total") \
    .start()

# Write hourly aggregation to console
hourly_query = hourly_total.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .queryName("hourly_total") \
    .start()

five_min_query.awaitTermination()
hourly_query.awaitTermination()


AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        

## Stream Simulation

In [23]:
kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f'{kafka_host}:9092')
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")
    .load()
)

In [24]:
from pyspark.sql.functions import from_json, col

parsed_df = (
    kafka_df
    .withColumn("value", expr("cast(value as string)"))
    .select(
        from_json(col("value"), schema)
        .alias("data")
    )
    .select("data.*")
)

NameError: name 'schema' is not defined

In [25]:
(
    parsed_df
    .writeStream
    .format("console")
    .outputMode("append")
    # .trigger(processingTime='5 seconds')
    # .trigger(continuous='1 second')
    # .trigger(once=true)
    .option("checkpointLocation", "checkpoint_dir")
    .start()
    .awaitTermination()
)

NameError: name 'parsed_df' is not defined